## 1. Initialize the project

Create the working context for the Project, if it has not already been created. The project serves as a placeholder for the code, data, and management of data operations inside the platform.

In [ ]:
!pip install digitalhub>=0.16 digitalhub-runtime-python digitalhub-runtime-modelserve

In [1]:
import digitalhub as dh
PROJECT_NAME = "rewind"
proj = dh.get_or_create_project(PROJECT_NAME)

Note: Make sure to replace <YOUR_PROJECT_NAME> with the actual name of your project before running the code.

## Prerequisites

Before running the dataset creation and fine-tuning jobs, create a Hugging Face access token and store it as a project secret named `HF_TOKEN`.

1. Open [Hugging Face Settings → Access Tokens](https://huggingface.co/settings/tokens).
2. Generate a new token.
3. Assign the required permissions for model and dataset access.
4. Copy the token value.
5. In the current project, create a secret named `HF_TOKEN` and paste the token as its value.

> The jobs below use `HF_TOKEN` to authenticate against Hugging Face. Do not hardcode the token in the notebook or source files.

## 2. Deploy Training Function


Fetch the "train" operation in the project. Train function is a python job that performs LLM fine tuning from a predefined HuggingFace model and dataset.

In [7]:
function_train = proj.get_function("train-llm")

In [8]:
image = function_train.refresh().spec.image
image

'harbor.atlas.fbk.eu/dev/dev-platform-rewind-train-llm:37cee'

In [9]:
function_train = proj.new_function(
    image=image,
    name="train-llm", 
    kind="python", 
    python_version="PYTHON3_11", 
    code_src="train.py",
    handler="train_and_log_model",
    requirements="requirements.txt"
)

Build the function as shown below.

In [7]:
func_b = function_train.run(action='build')

## 3. Perform Training
Training parameters define the model and dataset to be used. Training run refers to HF_TOKEN secret and expects a volume to be created of an appropriate size.

In [ ]:
profile = dh.get_k8s_resource_profiles()
profile

### Select a GPU profile

The `profile` list above contains the resource profiles available through the platform API. Select a GPU profile for the training job; this run uses `1xrtx5000`. Check that the selected profile is available to your project before submitting the job.

In [10]:
train_run = function_train.run(action="job",
                       profile="1xrtx5000",
                       parameters={
                         "model_id": "meta-llama/Llama-3.1-8B",
                         "model_name": "llmpa-tracked",
                         "hf_dataset_name": "team-bay/data-science-qa",
                         "train_data_path": "DataScienceDataset.csv",
                         "dev_data_path": "DataScienceDataset.csv",
                         "num_epochs": 2
                         },
                       secrets=["HF_TOKEN"],
                       envs=[
                          {"name": "HF_HOME", "value": "/app/local_data/huggingface"},
                          {"name": "TRANSFORMERS_CACHE", "value":  "/app/local_data/huggingface"}
                          ],
                       volumes=[{
                          "volume_type": "persistent_volume_claim",
                          "name": "volume-llmpa",
                          "mount_path": "/app/local_data",
                          "spec": { "size": "20Gi" }}]
					)

## 4. Serving the Model (REST API)

Deploy the fine-tuned model using DigitalHub's `vllmserve-text` runtime:

In [3]:
model = proj.get_model("llmpa-tracked")
model.key

'store://rewind/model/huggingface/llmpa-tracked:46ba5d3e02114ae5a4f53b3ce0fdc73a'

In [6]:
vllm_func = proj.new_function(
    name="vllm-text-serve",
    kind="vllmserve-text",
    model_name="llmpa-tracked",
    # engine='VLLM',
    url=model.key
)

In [ ]:
service_type: ClusterIP
task: vllmserve-text+serve://rewind/1974c548f8b545e6b128f635ea4f308c
adapters:
  - url: store://rewind/model/huggingface/llmpa-tracked:46ba5d3e02114ae5a4f53b3ce0fdc73a
    name: lora-adapted-llmpa-tracked
model_name: base-model
function: vllmserve-text://rewind/vllm-text-serve:2add8d4ca9de470ba872c59092f12633
profile: 1xrtx5000
url: store://rewind/model/huggingface/base_model:3jbqs70ns9p8b101mv7t8b0c
local_execution: false


In [7]:
serve_run = vllm_func.run(
    action="serve",
    profile="1xrtx5000",
    secrets=["HF_TOKEN"],
    wait=True
)

2026-10-14 07:34:06,501 - dhcore./home/khurshid/python3.12/lib/python3.12/site-packages/digitalhub_runtime_modelserve/entities/run/modelserve_run/entity.py - INFO - Waiting for run b01fcf2d3cf14b12ae47e715051de605 to deploy service.
2026-10-14 07:34:11,508 - dhcore./home/khurshid/python3.12/lib/python3.12/site-packages/digitalhub_runtime_modelserve/entities/run/modelserve_run/entity.py - INFO - Waiting for run b01fcf2d3cf14b12ae47e715051de605 to deploy service.
2026-10-14 07:34:16,520 - dhcore./home/khurshid/python3.12/lib/python3.12/site-packages/digitalhub_runtime_modelserve/entities/run/modelserve_run/entity.py - INFO - Waiting for run b01fcf2d3cf14b12ae47e715051de605 to deploy service.
2026-10-14 07:34:16,531 - dhcore./home/khurshid/python3.12/lib/python3.12/site-packages/digitalhub_runtime_modelserve/entities/run/modelserve_run/entity.py - INFO - Run b01fcf2d3cf14b12ae47e715051de605 service deployed.


### Inference Request Example

Once deployed, send POST requests to the serving endpoint:


In [ ]:
service = serve_run.refresh().status.service
model_name = serve_run.status.openai.get("model", "llmpa-tracked") if serve_run.status.openai else "llmpa-tracked"

payload = {
    "model": model_name,
    "prompt": "Explain gradient descent in machine learning.",
    "max_tokens": 256,
    "temperature": 0.7,
}

endpoint_url = f"http://{service['url']}/v1/completions" if service and "url" in service else None
response = serve_run.invoke(json=payload, url=endpoint_url).json()

print("Response from served model:")
print(response)